# HW5: Neural Density Estimation & Simulation-Based Inference

## Problem 1
Let's estimate the joint density of a synthetic exoplanet population catalog using a Masked Autoencoder for Distribution Estimation (MADE). 

The catalog has 4 properties per planet:
- $x_0=\log_{10}P$: period in days
- $x_1=\log_{10}R_p$: radius in $R_\oplus$ 
- $x_2$: an equilibrium-temperature-like proxy, a *nonlinear* function of $x_0$ plus noise
- $x_3$: host-star [Fe/H], heavy-tailed

$\log_{10} P$ and $\log_{10}R_p$ are drawn from a bimodal distribution ("radius valley"): close-in rocky planets and larger, gas-enveloped ones. 
Use the `make_synthetic_exoplanets` function to produce synthetic data

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import corner

torch.manual_seed(0)
device = "cpu"

def make_synthetic_exoplanets(N, device=device):
    '''synthetic exoplanet catalog: (log P, log Rp) radius-valley mixture,
    nonlinear temperature proxy, heavy-tailed metallicity
    '''
    mus = torch.stack([
        torch.tensor([0.3, 0.11], device=device),   # close-in, rocky (below the valley)
        torch.tensor([1.1, 0.38], device=device),    # wider orbit, gas-enveloped (above the valley)
    ])
    covs = torch.stack([
        torch.tensor([[0.06, 0.015], [0.015, 0.01]], device=device),
        torch.tensor([[0.10, -0.02], [-0.02, 0.02]], device=device),
    ])
    
    pis = torch.tensor([0.55, 0.45], device=device)
    comp_idx = torch.multinomial(pis, num_samples=N, replacement=True)
    mu, cov = mus[comp_idx], covs[comp_idx]
    eps = torch.randn(N, 2, device=device)
    L = torch.linalg.cholesky(cov)
    xy = (L @ eps.unsqueeze(-1)).squeeze(-1) + mu
    x0, x1 = xy[:, 0], xy[:, 1]     # log P, log Rp

    noise_x2 = 0.15*torch.randn(N, device=device)
    x2 = -0.6*x0 + 0.3*torch.sin(3*x0) + noise_x2      # Teq-like proxy, nonlinear in period

    nu = 4.0
    z = torch.randn(N, device=device)
    u = torch.distributions.Chi2(df=torch.tensor(nu, device=device)).sample((N,))
    x3 = -0.1 + 0.2*(z*torch.sqrt(nu/u))                # [Fe/H]-like, heavy-tailed

    return torch.stack([x0, x1, x2, x3], dim=-1)

**(a)** Generate $N=20{,}000$ samples with `make_synthetic_exoplanets` and make a corner plot. 
Identify the radius valley in the $(x_0,x_1)$ panel. 
What makes $x_2$ and $x_3$ hard for a simple Gaussian mixture model to capture? 

**(b)** Implement a MADE to estimate $p(\bf{x})$, as in the lecture. 
Your MADE object should include 
* `log_prob(x)` to compute $\log p(\bf{x})$
* `sample(num_samples)`: sample $\bf{x}' \sim p(\bf{x})$

**(c)** Train your MADE on the exoplanet catalog. 
Plot the training loss curve. 
Then draw samples from the trained model and overlay a corner plot of the model samples on the true catalog. 
Does it capture the radius valley? The nonlinear $x_0$-$x_2$ coupling? The heavy tail in $x_3$?

*Hint: Adam optimizer, a few hundred epochs, minimize negative mean log-likelihood*

**(d)** Evaluating `log_prob` only requires a single forward pass, but `sample` requires $D=4$ sequential forward passes. Why? 

## Problem 2
In HW4 Problem 1, you used ABC to infer the SN rate, $\lambda$, from a magnitude-limited survey with a logisti function — a case where the likelihood is intractable but the simulator is cheap. 
In HW4 Problem 3, you sped this up by fitting a VI surrogate to the joint $p(\lambda,\mathcal D)$ and running ABC on top of it.

Now let's use neural posterior estimation (NPE) via the `sbi` package to learn $p(\lambda\,|\,\mathcal D)$ directly.

**(a)** Recreate the HW4 Problem 1 simulator: $\lambda\sim\mathcal U(50,500)$; $N_{\rm true}\sim{\rm Poisson}(\lambda)$; each event has $m_{\rm true}\sim\mathcal U(18,24)$, $m_{\rm obs}=m_{\rm true}+\mathcal N(0,\sigma_M{=}0.1)$; detected with probability $c(m)=1/(1+\exp[(m-23)/0.3])$. Use $\mathcal D=(N_{\rm obs},\bar m_{\rm obs})$ as your data vector. 
Generate one "observed" dataset with $\lambda_{\rm true}=250$.

**(b)** Draw $\gtrsim2\times10^4$ pairs $(\lambda',\mathcal D')$. Using `sbi.inference.NPE` with a `'maf'` density estimator, train a neural posterior estimator and sample the posterior for your observed $\mathcal D$.

**(c)** Compare the mean/std of your NPE posterior on $\lambda$ to (i) your HW4 Problem 1(c) ABC posterior and (ii) your HW4 Problem 3(b) VI-surrogate+ABC posterior. Do all three roughly agree?

**(d)** What are the trade-offs of NPE? 
What advantage does NPE have over ABC and VI+ABC? 
What did you have to trust instead? 

*Hint: How would you know if `.train()` hadn't actually converged?*

## Problem 3
An NPE posterior can look reasonable on a corner plot even when it's wrong.
Let's rigorously validate the NPE from Problem 2 using simulation-based calibration (SBC), a probability-probability (PP) plot, and TARP coverage.

**(a)** Draw $\gtrsim10^3$ pairs $(\lambda_{\rm test}',\mathcal D_{\rm test}')$ for your test sample. 
For each $D'_{\rm test}$ sample the posterior $p(\lambda | D'_{\rm test})$ using your trained NPE. 

**(b)** Simulation-based calibration: For each $(\lambda_{\rm test}',\mathcal D_{\rm test}')$, compute the rank of $\lambda{\rm test}'$ within the posterior samples. 
Plot the rank histogram. 

**(c)** From the same ranks, make a probability-probability (PP) plot: expected coverage (the diagonal) vs. the empirical CDF of the (normalized) ranks

**(d)** Now validate the posterior with TARP coverage, using the `tarp` package. 
Plot expected versus estimated coverage.

**(e)** Based on the SBC, PP plot, and TARP coverage, are the posteriors estimated by your NPE well calibrated? 
If not, what are the limitations of the NPE? 

*Hint: Is it biased? Do the posteriors accurately quantify the uncertainties?*

## Problem 4
Let's push NPE onto a higher-dimensiona and see whether it holds up.

Let's go back to the radial-velocity model from HW4,
$$v(t) = v_0 + K\sin\left(\frac{2\pi t}{P}+\phi\right),$$
with the same ground-based nightly-cadence setup. This time, 
$\theta=(v_0,K,P,\phi)$ is 4-dimensional, and one of those dimensions ($P$) is multimodal.

**(a)** Recreate the HW4 Problem 2 simulator: using `np.random.RandomState(50)`, 18 of 40 nights, $t={\rm night}+0.5+0.05\,\mathcal N(0,1)$; $v_0=0$, $K=12$ m/s, $P_{\rm true}=4.2$ days, $\phi=0.8$ rad, $\sigma_{\rm RV}=2$ m/s. 
Use priors 
$$v_0\sim\mathcal U(-10,10)$$ 
$$K\sim\mathcal U(0,30)$$ 
$$P\sim\mathcal U(0.5,15)$$ 
$$\phi\sim\mathcal U(0,2\pi)$$
Generate the observed RV data $v_{\rm obs}$ (the full 18-point vector). 

**(b)** Generate a training data set with $2\times10^4$ simulations and then train an NPE (`'maf'`). 
Sample the posterior for $v_{\rm obs}$ and make a corner plot of $(v_0,K,P,\phi)$.

**(c)** Run SBC separately for all 4 parameters using $\gtrsim10^3$ separately generated test sample and plot all 4 rank histograms. 
Do all 4 parameters look calibrated?

**(d)** Run TARP on the full 4D joint posterior and plot the expected versus estimated coverage.
What the coverage reveal? How does that differ or agree with the SBC run on each parameter individually?

**(e)** Suppose you have a fixed simulation bugdet (i.e. you are not allowed to run more simulations). 
Discuss how could you restructure how you spend the same fixed number of simulations to get a better-calibrated posterior for this specific observed dataset?

*Hint: Most of that budget was spent on prior draws of $P$ spread across the full $[0.5,15]$ day range*